# Run the local VLM arm on a free Colab T4 (IT-98)

The dual-extractor pipeline needs two independent readings of each card.
Gemini runs anywhere; `qwen2.5-vl` needs a real GPU, which nobody's laptop
has. This notebook is that GPU.

It produces one thing: a `raw/` directory of the model's answers, written
into the shared Drive. `benchmark/dual_extract.py` pairs it against Gemini's on
any machine afterwards — the pairing reads directories, not live models, so
the two arms never have to run in the same place.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Free tier gives ~12h per session and disconnects after ~90 min idle. Thirty
cards is well inside both, but leave the tab open.

In [ ]:
# Fail now, not eight minutes into a model download, if there is no GPU.
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}, {vram:.0f} GB, bf16 supported: {torch.cuda.is_bf16_supported()}")
print(
    "-> compute dtype will be",
    "bfloat16" if torch.cuda.is_bf16_supported() else "float16",
)

In [ ]:
# Colab ships torch; the rest is ours. accelerate is what device_map="auto"
# needs, bitsandbytes is the 4-bit loader.
%pip install -q accelerate bitsandbytes qwen-vl-utils openpyxl pypdf "transformers>=4.57"

## The repo

Private, so the clone needs a token. `getpass` keeps it off the screen and
out of the saved notebook — **do not** paste a token into a cell instead.
Use a fine-grained PAT with read-only contents access, and delete it when
you are done.

The token is used once, for the clone, and not written anywhere.

In [ ]:
import getpass
import subprocess
from pathlib import Path

REPO = "skandaramanan/IT-Project-T8-6"
BRANCH = "main"
checkout = Path("/content/IT-Project-T8-6")

if not checkout.exists():
    token = getpass.getpass("GitHub token (input hidden): ")
    # Output is captured rather than left to stream, and the token is redacted
    # out of anything raised: check=True would put the whole token-bearing URL
    # into the CalledProcessError message, printed straight into the notebook.
    result = subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            BRANCH,
            f"https://{token}@github.com/{REPO}.git",
            str(checkout),
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.returncode:
        raise RuntimeError(
            (result.stderr or "git clone failed").replace(token, "***").strip()
        )
    # Scrub the token from .git/config (git stores the clone URL as origin).
    subprocess.run(
        [
            "git",
            "-C",
            str(checkout),
            "remote",
            "set-url",
            "origin",
            f"https://github.com/{REPO}.git",
        ],
        check=True,
    )
    del token
print("checked out:", checkout)

## The data — upload this once, first

The Drive holds the **raw scans** (`Wilfred_Prest_Survey_Data/`, the numbered
unit folders). It does **not** hold the C1/C2 benchmark set — the 30 selected
cards and their hand-checked sheet live only on whoever ran
`select_c1_c2_sample.py`.

So before running this, copy your local `data/benchmark/c1c2/` (146 MB: 30
PNGs, the ground-truth xlsx, `manifest.csv`) into Drive. Keep the folder
named `c1c2` and put it anywhere — the next cell finds it. Both arms must
score the same 30 cards, or their agreement measures nothing.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Set this only if the search below finds nothing, or finds several.
DRIVE_ROOT = None

if DRIVE_ROOT is None:
    # Bounded search rather than a full walk -- Drive over FUSE is slow, and
    # the folder is never more than a few levels down.
    hits = [
        d
        for depth in range(6)
        for d in Path("/content/drive").glob("/".join(["*"] * depth + ["c1c2"]))
        if d.is_dir()
    ]
    if len(hits) == 1:
        DRIVE_ROOT = hits[0].parent
        print(f"found c1c2 in {DRIVE_ROOT}")
    elif hits:
        raise SystemExit(
            "Several c1c2 folders:\n  "
            + "\n  ".join(str(h) for h in hits)
            + "\nSet DRIVE_ROOT above to the parent of the one you want."
        )
    else:
        top = sorted(p.name for p in Path("/content/drive").glob("*/*") if p.is_dir())
        raise SystemExit(
            "No folder named c1c2 anywhere in your Drive.\n"
            "Upload your local data/benchmark/c1c2/ (keep that name).\n"
            f"Top-level folders I can see: {top[:25]}"
        )

DRIVE_ROOT = Path(DRIVE_ROOT)
BENCHMARK = DRIVE_ROOT / "c1c2"
GROUND_TRUTH = BENCHMARK / "coding-card-c1-c2-ground-truth.xlsx"
IMAGES = BENCHMARK

# Results go back to Drive, not /content -- Colab's disk dies with the
# session, and this raw/ directory is the entire point of the run.
RESULTS_ROOT = DRIVE_ROOT / "results"

if not GROUND_TRUTH.exists():
    raise SystemExit(
        f"Found {BENCHMARK}, but no {GROUND_TRUTH.name} inside it.\n"
        f"It holds: {sorted(p.name for p in BENCHMARK.iterdir())[:8]}"
    )

print(f"sheet   : {GROUND_TRUTH}")
print(f"images  : {len(list(IMAGES.glob('*.png')))} png (expect 30)")
print(f"results : {RESULTS_ROOT}")

## One card first

The model is a ~16GB download and the first card pays for it. Run one before
committing to thirty: it proves the weights load 4-bit, the image fits, and
the JSON parses. If this OOMs, drop `MAX_PIXELS` in
`backend/survey/extraction/qwen.py` rather than reaching for a bigger GPU.

In [ ]:
%cd /content/IT-Project-T8-6
# Installs the survey package the harness imports. --no-deps leaves Colab's
# torch stack alone: cell 2 already installed what the Qwen arm needs.
# --ignore-requires-python because Colab's Python need not be the pinned 3.12.
%pip install -q --no-deps --ignore-requires-python -e .
# The harness isn't part of the package. Working from its folder puts it on
# the import path, the same way running one of its scripts does.
%cd benchmark
from ground_truth import load_ground_truth

first = load_ground_truth(GROUND_TRUTH, IMAGES)[0]
print("first card:", first.page_id, "->", first.image_path.name)

!python run_benchmark.py --model qwen2.5-vl \
  --ground-truth "{GROUND_TRUTH}" --images "{IMAGES}" \
  --results-root "{RESULTS_ROOT}" --workers 1 --pages {first.page_id}

## The rest

`--workers 1` because the GPU is the bottleneck, not the network — parallel
workers on one card would just fight over VRAM.

Cards already answered are skipped, so a disconnect costs only what was in
flight. Re-run this cell to pick up where it stopped.

In [ ]:
!python run_benchmark.py --model qwen2.5-vl \
  --ground-truth "{GROUND_TRUTH}" --images "{IMAGES}" \
  --results-root "{RESULTS_ROOT}" --workers 1

In [ ]:
# What made it into Drive. This is the artefact -- the accuracy number above
# is IT-100's business, not this notebook's.
raw = RESULTS_ROOT / "qwen2.5-vl" / "raw"
answered = [p for p in sorted(raw.glob("*.txt")) if p.stat().st_size]
expected = {c.page_id for c in load_ground_truth(GROUND_TRUTH, IMAGES)}

print(f"{len(answered)}/{len(expected)} cards answered, saved to {raw}")
missing = sorted(expected - {p.stem for p in answered})
print("still unread:", missing or "none")

## Back on your own machine

Sync the Drive `results/` folder down, then pair the two arms:

```bash
uv run python benchmark/dual_extract.py --models gemini qwen2.5-vl --no-fetch
```

`--no-fetch` because both arms are already extracted — one here, one there —
and there is nothing left to buy.